# 使用Video MAE进行快速推理



## 1. Load video 加载视频

让我们从[Kinetics-400数据集](https://www.deepmind.com/open-source/kinetics)中加载一个视频。该数据集包含数百万个YouTube视频，每个视频都被标注为400种可能类别中的一种。

In [ ]:
!wget https://huggingface.co/datasets/nielsr/video-demo/resolve/main/eating_spaghetti.mp4

In [ ]:
from ipywidgets import Video

video_path = "eating_spaghetti.mp4" 
Video.from_file(video_path, width=500)

## 2. Prepare video for model 为模型准备视频

我们可以通过使用VideoMAEFeatureExtractor为模型处理视频。首先，从最多300帧中采样16帧，并将这些帧输入特征提取器。

它将进行一些基本的预处理操作，包括对视频的每一帧进行调整大小、中心裁剪以及归一化处理。

In [ ]:
from mindnlp.transformers import VideoMAEFeatureExtractor

feature_extractor = VideoMAEFeatureExtractor.from_pretrained("MCG-NJU/videomae-base-finetuned-kinetics")

In [ ]:
from decord import VideoReader, cpu
import numpy as np

# video clip consists of 300 frames (10 seconds at 30 FPS)
vr = VideoReader(video_path, num_threads=1, ctx=cpu(0)) 

def sample_frame_indices(clip_len, frame_sample_rate, seg_len):
  converted_len = int(clip_len * frame_sample_rate)
  end_idx = np.random.randint(converted_len, seg_len)
  str_idx = end_idx - converted_len
  index = np.linspace(str_idx, end_idx, num=clip_len)
  index = np.clip(index, str_idx, end_idx - 1).astype(np.int64)
  
  return index

vr.seek(0)
index = sample_frame_indices(clip_len=16, frame_sample_rate=4, seg_len=len(vr))
buffer = vr.get_batch(index).asnumpy()
buffer.shape

In [ ]:
# create a list of NumPy arrays
video = [buffer[i] for i in range(buffer.shape[0])]

encoding = feature_extractor(video, return_tensors="ms")
print(encoding.pixel_values.shape)

## 3. Load model 加载模型

接下来，让我们从hub中加载模型。

In [ ]:
from mindnlp.transformers import VideoMAEFeatureExtractor, VideoMAEForVideoClassification

model = VideoMAEForVideoClassification.from_pretrained("MCG-NJU/videomae-large-finetuned-kinetics")


## 4. Forward pass 前向传播

In [ ]:
pixel_values = encoding.pixel_values

# forward pass
outputs = model(pixel_values)
logits = outputs.logits

In [ ]:
predicted_class_idx = logits.argmax(-1).item()

print("Predicted class:", model.config.id2label[predicted_class_idx])